# LLM Benchmark — GPT vs Claude vs Gemini on Databricks

Sends identical prompts to all providers via **Databricks AI Gateway** and auto-validates each response using purpose-built validators:
- **Numeric** — extracts number and checks within tolerance
- **Code execution** — runs generated Python against test cases
- **JSON schema** — parses JSON and validates keys + values
- **Regex extraction** — verifies all expected items are found and well-formed
- **Classification** — checks every item maps to the correct label
- **SQL** — checks required clauses and rejects forbidden patterns


## 1 · Setup

In [ ]:
# %pip install openai pandas matplotlib -q

In [ ]:
import os, sys

PROJECT_ROOT = os.path.abspath(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.environ.setdefault("DATABRICKS_HOST",  "https://adb-<workspace-id>.azuredatabricks.net")
os.environ.setdefault("DATABRICKS_TOKEN", "dapi<your-personal-access-token>")

print("Host :", os.environ["DATABRICKS_HOST"])
print("Token:", os.environ["DATABRICKS_TOKEN"][:8] + "…")

## 2 · Models & pricing

In [ ]:
import pandas as pd
from src.llm_benchmark.config import MODELS

df_models = pd.DataFrame(MODELS)[
    ["display_name", "provider", "input_price_per_1m", "output_price_per_1m"]
].rename(columns={
    "display_name":        "Model",
    "provider":            "Provider",
    "input_price_per_1m":  "Input $/1M tokens",
    "output_price_per_1m": "Output $/1M tokens",
})

df_models.style \
    .format({"Input $/1M tokens": "${:.3f}", "Output $/1M tokens": "${:.3f}"}) \
    .set_caption("Vendor AI token prices (no DBU costs)") \
    .hide(axis="index")

## 3 · Test prompts & validators

In [ ]:
from src.llm_benchmark.prompts import TEST_PROMPTS

df_prompts = pd.DataFrame(
    [{"id": p["id"], "category": p["category"],
      "validator_type": p["validator"].__name__ if hasattr(p["validator"], "__name__") else type(p["validator"]).__name__,
      "prompt": p["prompt"][:80] + "…" if len(p["prompt"]) > 80 else p["prompt"]}
     for p in TEST_PROMPTS]
).rename(columns={"id": "ID", "category": "Category",
                  "validator_type": "Validator", "prompt": "Prompt (truncated)"})

df_prompts.style.set_properties(**{"text-align": "left"}).hide(axis="index")

## 4 · Run benchmark

In [ ]:
from src.llm_benchmark.benchmark import run_benchmark

# Change to test a subset: ["openai"], ["anthropic"], ["google"]
selected_providers = ["openai", "anthropic", "google"]
models_to_run = [m for m in MODELS if m["provider"] in selected_providers]

print(f"Running {len(models_to_run)} models × {len(TEST_PROMPTS)} prompts "
      f"= {len(models_to_run) * len(TEST_PROMPTS)} API calls …\n")

results = run_benchmark(models=models_to_run, max_tokens=512)
print(f"\nDone — {len(results)} results collected.")

## 5 · Detail results

In [ ]:
from dataclasses import asdict

df = pd.DataFrame([asdict(r) for r in results])

detail_cols = [
    "model_display_name", "prompt_id", "prompt_category",
    "input_tokens", "output_tokens", "total_tokens",
    "ai_cost_usd", "response_time_sec",
    "accuracy_label", "accuracy_score", "accuracy_detail",
]

df_detail = df[detail_cols].rename(columns={
    "model_display_name": "Model",
    "prompt_id":          "Prompt",
    "prompt_category":    "Category",
    "input_tokens":       "In Tok",
    "output_tokens":      "Out Tok",
    "total_tokens":       "Total Tok",
    "ai_cost_usd":        "AI Cost ($)",
    "response_time_sec":  "Time (s)",
    "accuracy_label":     "Result",
    "accuracy_score":     "Score",
    "accuracy_detail":    "Why",
})

ACCURACY_COLORS = {
    "Excellent": "background-color: #c6efce",
    "Good":      "background-color: #ffeb9c",
    "Partial":   "background-color: #ffcc99",
    "Poor":      "background-color: #ffc7ce",
    "Error":     "background-color: #d9d9d9",
}

df_detail.style \
    .format({"AI Cost ($)": "${:.6f}", "Time (s)": "{:.2f}s", "Score": "{:.2f}"}) \
    .applymap(lambda v: ACCURACY_COLORS.get(v, ""), subset=["Result"]) \
    .set_caption("Detail: every model × every prompt with validator feedback") \
    .hide(axis="index")

## 6 · Summary per model

In [ ]:
summary = (
    df.groupby("model_display_name")
    .agg(
        Prompts       = ("prompt_id",        "count"),
        Errors        = ("error",             lambda x: x.notna().sum()),
        Avg_Tokens    = ("total_tokens",      "mean"),
        Total_AI_Cost = ("ai_cost_usd",       "sum"),
        Avg_Time_sec  = ("response_time_sec", "mean"),
        Avg_Accuracy  = ("accuracy_score",    "mean"),
    )
    .reset_index()
    .rename(columns={"model_display_name": "Model"})
    .sort_values("Avg_Accuracy", ascending=False)
    .reset_index(drop=True)
)

summary.style \
    .format({
        "Avg_Tokens":    "{:.0f}",
        "Total_AI_Cost": "${:.6f}",
        "Avg_Time_sec":  "{:.2f}s",
        "Avg_Accuracy":  "{:.2f}",
    }) \
    .background_gradient(subset=["Avg_Accuracy"], cmap="RdYlGn") \
    .background_gradient(subset=["Total_AI_Cost"], cmap="RdYlGn_r") \
    .background_gradient(subset=["Avg_Time_sec"], cmap="RdYlGn_r") \
    .set_caption("Summary: aggregated per model (sorted by accuracy)") \
    .hide(axis="index")

## 7 · Charts

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("LLM Benchmark — Databricks AI Gateway", fontsize=14, fontweight="bold")

model_names = summary["Model"]
x = range(len(model_names))
colors = ["#4472C4", "#ED7D31", "#A9D18E", "#FFC000", "#5B9BD5", "#FF6B6B", "#C55A11", "#538135"]

for ax, col, title, ylabel, fmt in [
    (axes[0], "Avg_Accuracy",  "Avg Accuracy Score",   "Score (0–1)", "{:.2f}"),
    (axes[1], "Total_AI_Cost", "Total AI Cost (USD)",  "USD",         "${:.5f}"),
    (axes[2], "Avg_Time_sec",  "Avg Response Time (s)", "Seconds",    "{:.2f}s"),
]:
    vals = summary[col]
    ax.bar(x, vals, color=colors[:len(model_names)])
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=35, ha="right", fontsize=8)
    ax.set_ylabel(ylabel)
    for i, v in enumerate(vals):
        ax.text(i, v * 1.02, fmt.format(v), ha="center", fontsize=8)

axes[0].set_ylim(0, 1.15)
plt.tight_layout()
plt.savefig("llm_benchmark_charts.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved → llm_benchmark_charts.png")

## 8 · Accuracy heatmap — by category × model

In [ ]:
pivot = df.pivot_table(
    index="prompt_category",
    columns="model_display_name",
    values="accuracy_score",
    aggfunc="mean",
)

pivot.style \
    .format("{:.2f}") \
    .background_gradient(cmap="RdYlGn", axis=None) \
    .set_caption("Avg accuracy score by category × model  (green = best)")

## 9 · Validator failure drill-down

Shows every row where accuracy was not Excellent — includes the **Why** column from the validator.

In [ ]:
failures = df_detail[df_detail["Result"] != "Excellent"].copy()

if failures.empty:
    print("All responses scored Excellent!")
else:
    failures.style \
        .format({"AI Cost ($)": "${:.6f}", "Time (s)": "{:.2f}s", "Score": "{:.2f}"}) \
        .applymap(lambda v: ACCURACY_COLORS.get(v, ""), subset=["Result"]) \
        .set_caption(f"{len(failures)} non-Excellent responses — validator detail shown in 'Why'") \
        .hide(axis="index")

## 10 · Inspect raw response for any model + prompt

In [ ]:
# Prompt IDs: capital_france, compound_interest, prime_check_math,
#             code_reverse_string, code_fibonacci, code_word_count,
#             json_person_extraction, json_product_list,
#             extract_emails, sentiment_multi, topic_classification,
#             sql_aggregation, sql_window_function

INSPECT_MODEL  = "GPT-4o"
INSPECT_PROMPT = "code_fibonacci"

row = df[(df["model_display_name"] == INSPECT_MODEL) & (df["prompt_id"] == INSPECT_PROMPT)]

if row.empty:
    print("No match — check INSPECT_MODEL and INSPECT_PROMPT values.")
else:
    r = row.iloc[0]
    print(f"Model           : {r['model_display_name']}")
    print(f"Prompt          : {r['prompt_text']}")
    print(f"\n--- Response ---\n{r['response_text']}")
    print(f"\nTokens          : {r['input_tokens']} in / {r['output_tokens']} out")
    print(f"AI Cost         : ${r['ai_cost_usd']:.6f}")
    print(f"Response Time   : {r['response_time_sec']:.2f}s")
    print(f"Accuracy        : {r['accuracy_label']} (score={r['accuracy_score']:.2f})")
    print(f"Validator detail: {r['accuracy_detail']}")

## 11 · Export to CSV

In [ ]:
csv_path = "llm_benchmark_results.csv"
df[detail_cols].to_csv(csv_path, index=False)
print(f"Saved {len(df)} rows → {csv_path}")